In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

En este año tenemos dos encuestas diferentes, una hecha a septiembre y otra a diciembre, calculamos en función de las dos encuestas para hacer más específicos los cálculos

In [2]:
# data = pd.read_stata(r"Z:\harmonized\ECU\ENEMDU\data_arm\ECU_1990m11_BID.dta") # para bases de stata
data9 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2003/m9/data_orig/ecu03-sep.dta", convert_categoricals=False) # para bases de stata
data12 = pd.read_stata(r"/home/edu/Dropbox/datos/ECU/ECU/ENEMDU/2003/m12/data_orig/per12_2003.dta", convert_categoricals=False) # para bases de stata

## Revisar los datos

### Septiembre
- rn
- ciudad
- zona
- sector
- panelm
- vivienda
- hogar
- persona
- numpers
- edad
- fexp

- pe63

### Diemebre
- rgnal
- rn
- area
- prov
- ciudad
- zona
- sector
- panelm
- vivienda
- hogar
- numpers
- persona
- edad
- fexp

- pe63

En estas encuestas mantenemos pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

Hay variables dicotómicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, se puede usar estas variables y el ingreso laboral asumiendo que cuando estaba trabajando tenía ese ingreso para intentar aproximar el salario mensual y de ahí el salario trimestral, esto solo funciona así ya que no tenemos una variable que explicite el mes, en encuestas que tengan el mes o trimestre explícito esto no sería igual.

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [3]:
data9 = data9[['rn', 'ciudad', 'zona', 'sector', 'panelm', 'vivienda', 'hogar',
               'persona', 'numpers', 'edad', 'fexp', 'pe63',
               'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 
              'oct', 'nov', 'dic']]

data12 = data12[['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 
                    'panelm', 'vivienda', 'hogar', 'persona', 'numpers', 'edad',
                     'fexp', 'pe63', 
                     'ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 
              'oct', 'nov', 'dic']]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [4]:
data9['pe63'] = data9['pe63'].replace(25015, np.nan)
data9['pe63'] = data9['pe63'].replace(160000, np.nan)
data9['pe63'] = data9['pe63'].replace(999999, np.nan)

data12['pe63'] = data12['pe63'].replace(25015, np.nan)
data12['pe63'] = data12['pe63'].replace(160000, np.nan)
data12['pe63'] = data12['pe63'].replace(999999, np.nan)

In [5]:
data9['ingr'] = data9['pe63']
data12['ingr'] = data12['pe63']

Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados en un mes, ahora las variables categoricas por mes tienen diferentes leyendas y no aparecen las etiquetas en la base de 2003, planteamos estas etiquetas basandonos en la continuidad más lógica desde diciembre de 2002, hacemos esto solo para la base de septiembre, para encontrar los dos primeros trimestres, en cuanto a la base de diciembre la tomamos como el valor del cuarto trimestre

En diciembre de 2002 las personas reportadas como Trabajando fueron 6197, en enero de 2003 son 24725
- 1 - Trabajando
- 2 - Buscando trabajo
- 3 - Desocupado

In [6]:
data9['ingr_ene'] = data9.apply(lambda x: x['ingr'] if x['ene'] == 1 else None, axis=1)
data9['ingr_feb'] = data9.apply(lambda x: x['ingr'] if x['feb'] == 1 else None, axis=1)
data9['ingr_mar'] = data9.apply(lambda x: x['ingr'] if x['mar'] == 1 else None, axis=1)
data9['ingr_abr'] = data9.apply(lambda x: x['ingr'] if x['abr'] == 1 else None, axis=1)
data9['ingr_may'] = data9.apply(lambda x: x['ingr'] if x['may'] == 1 else None, axis=1)
data9['ingr_jun'] = data9.apply(lambda x: x['ingr'] if x['jun'] == 1 else None, axis=1)
data9['ingr_jul'] = data9.apply(lambda x: x['ingr'] if x['jul'] == 1 else None, axis=1)
data9['ingr_ago'] = data9.apply(lambda x: x['ingr'] if x['ago'] == 1 else None, axis=1)
data9['ingr_sep'] = data9.apply(lambda x: x['ingr'] if x['sep'] == 1 else None, axis=1)

data12['ingr_oct'] = data12.apply(lambda x: x['ingr'] if x['oct'] == 1 else None, axis=1)
data12['ingr_nov'] = data12.apply(lambda x: x['ingr'] if x['nov'] == 1 else None, axis=1)
data12['ingr_dic'] = data12.apply(lambda x: x['ingr'] if x['dic'] == 1 else None, axis=1)

In [7]:
data9[['ingr_ene', 'ingr_feb', 'ingr_mar', 'ingr_abr', 'ingr_may', 'ingr_jun', 'ingr_jul', 'ingr_ago', 'ingr_sep']].mean()

ingr_ene    114.283921
ingr_feb    114.441870
ingr_mar    114.701236
ingr_abr    115.289715
ingr_may    115.411885
ingr_jun    115.267869
ingr_jul    114.939756
ingr_ago    115.355078
ingr_sep    115.826485
dtype: float64

In [8]:
data12[['ingr_oct', 'ingr_nov', 'ingr_dic']].mean()

ingr_oct    166.245448
ingr_nov    168.709599
ingr_dic    182.114772
dtype: float64

## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [9]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2003]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [10]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Guayaquil': fila['Guayaquil'],
        'Quito': fila['Quito'],
        'Cuenca': fila['Cuenca']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [11]:
# Corregimos los códigos para usarlos cómo texto
data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

Diccionario ciudades disponibles

In [12]:
parroquia_dict = {
    '01': 'Cuenca',
    '09': 'Guayaquil',
    '17': 'Quito'
}

data9['ciudad_asignada'] = data9['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))
data12['ciudad_asignada'] = data12['ciudad_2'].apply(lambda x: parroquia_dict.get(x, 'Nacional'))

In [13]:
data9['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     10797
Guayaquil     6960
Quito         4822
Cuenca        2032
Name: count, dtype: int64

In [14]:
data12['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional     61309
Guayaquil    10055
Quito         6869
Cuenca        4084
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [15]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [16]:
data9['ipc_t1'] = data9.apply(lambda fila: asigna_ipc(fila, 1), axis=1)
data9['ipc_base_t1'] = data9.apply(lambda fila: asigna_ipc_base(fila, 1), axis=1)

data9['ipc_t2'] = data9.apply(lambda fila: asigna_ipc(fila, 2), axis=1)
data9['ipc_base_t2'] = data9.apply(lambda fila: asigna_ipc_base(fila, 2), axis=1)

data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

In [17]:
# Calculamos el deflactor
data9['def_t1'] = (data9['ipc_base_t1'] / data9['ipc_t1'])
data9['def_t2'] = (data9['ipc_base_t2'] / data9['ipc_t2'])
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])

data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

In [18]:
# Ingreso real por mes
data9['ingr_ene_r'] = data9['ingr_ene'] * data9['def_t1']
data9['ingr_feb_r'] = data9['ingr_feb'] * data9['def_t1']
data9['ingr_mar_r'] = data9['ingr_mar'] * data9['def_t1']
data9['ingr_abr_r'] = data9['ingr_abr'] * data9['def_t2']
data9['ingr_may_r'] = data9['ingr_may'] * data9['def_t2']
data9['ingr_jun_r'] = data9['ingr_jun'] * data9['def_t2']
data9['ingr_jul_r'] = data9['ingr_jul'] * data9['def_t3']
data9['ingr_ago_r'] = data9['ingr_ago'] * data9['def_t3']
data9['ingr_sep_r'] = data9['ingr_sep'] * data9['def_t3']

data12['ingr_oct_r'] = data12['ingr_oct'] * data12['def_t4']
data12['ingr_nov_r'] = data12['ingr_nov'] * data12['def_t4']
data12['ingr_dic_r'] = data12['ingr_dic'] * data12['def_t4']

Ingreso mensual promedio en el trimeste

In [19]:
data9['ingr_t1_r'] = (data9['ingr_ene_r'] + data9['ingr_feb_r'] + data9['ingr_mar_r'])/3
data9['ingr_t2_r'] = (data9['ingr_abr_r'] + data9['ingr_may_r'] + data9['ingr_jun_r'])/3
data9['ingr_t3_r'] = (data9['ingr_jul_r'] + data9['ingr_ago_r'] + data9['ingr_sep_r'])/3

data12['ingr_t4_r'] = (data12['ingr_oct_r'] + data12['ingr_nov_r'] + data12['ingr_dic_r'])/3

In [20]:
data9[['ingr_t1_r', 'ingr_t2_r', 'ingr_t3_r']].mean()

ingr_t1_r    171.316764
ingr_t2_r    174.930521
ingr_t3_r    177.514720
dtype: float64

In [21]:
data12[['ingr_t4_r']].mean()

ingr_t4_r    261.741478
dtype: float64

## Regiones

In [22]:
# Corregimos los códigos para usarlos cómo texto
data9['ciudad'] = data9['ciudad'].apply(str)
data9['ciudad'] = data9['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data9['ciudad_2'] = data9['ciudad'].apply(lambda x: x[:2])

data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)

data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:2])

In [23]:
regiones_dict = {
    'Guayas': '09',
    'Manabí': '13',
    'El Oro': '07',
    'Los Ríos': '12',
    'Pichincha': '17',
    'Azuay': '01',
    'Galápagos': '20',
    'Sierra': ['04', '10', '05', '18', '02', '06', '03', '11'],
    'Costa, Santo Domingo': ['08', '24', '23'],
    'Amazonía': ['14', '15', '16', '19', '21', '22', '90']
}

In [24]:
codigo_region = {}
for region, codes in regiones_dict.items():
    
    if isinstance(codes, list):
        for code in codes:
            codigo_region[code] = region
    
    else:
        codigo_region[codes] = region

# Mapeo de regiones
data9['region'] = data9['ciudad_2'].map(codigo_region)

data12['region'] = data12['ciudad_2'].map(codigo_region)

In [25]:
data9['region'].value_counts()

region
Guayas                  6960
Pichincha               4822
Sierra                  2923
El Oro                  2449
Amazonía                2139
Azuay                   2032
Manabí                  1579
Los Ríos                1031
Costa, Santo Domingo     676
Name: count, dtype: int64

In [27]:
data12['region'].value_counts()

region
Sierra                  31338
Guayas                  10055
Pichincha                6869
Costa, Santo Domingo     6544
Manabí                   6490
Los Ríos                 6137
El Oro                   6075
Amazonía                 4725
Azuay                    4084
Name: count, dtype: int64

## Calculo ingreso de los hogares

In [28]:
columnas_idef = ['rn', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data9['idef_hogar'] = data9[columnas_idef].astype(str).agg(''.join, axis=1)
len(data9['idef_hogar'].unique())

1560

In [29]:
columnas_idef = ['rgnal', 'rn', 'area', 'prov', 'ciudad', 'zona', 'sector', 'vivienda', 'hogar']

data12['idef_hogar'] = data12[columnas_idef].astype(str).agg(''.join, axis=1)
len(data12['idef_hogar'].unique())

4998

Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [30]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [31]:
data9['ingr_t1_h'] = data9.groupby('idef_hogar')['ingr_t1_r'].transform(sum_with_na)
data9['ingr_t2_h'] = data9.groupby('idef_hogar')['ingr_t2_r'].transform(sum_with_na)
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)

data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

In [32]:
data9[['ingr_t1_h', 'ingr_t2_h', 'ingr_t3_h']].mean()

ingr_t1_h    1038.090463
ingr_t2_h    1050.088097
ingr_t3_h    1073.993781
dtype: object

In [33]:
data12[['ingr_t4_h']].mean()

ingr_t4_h    907.070961
dtype: object

In [34]:
print("Ingreso medio de un hogar t4: ", data12['ingr_t4_h'].mean())
print("Mediana del ingreso de un hogar t4: ", data12['ingr_t4_h'].median())

Ingreso medio de un hogar t4:  907.0709614939261
Mediana del ingreso de un hogar t4:  694.5835559940576


## Sacamos edades negativas y mayores a 100 años

In [35]:
len(data9)

24611

In [36]:
len(data12)

82317

En este caso en la variable edad tenemos números y el texto 'menos de un año' así que primero transformamos todas las filas que digan 'menos de un año' a 0

In [37]:
data9['edad'] = data9['edad'].apply(lambda x: x if type(x) == int else 0)
data12['edad'] = data12['edad'].apply(lambda x: x if type(x) == int else 0)

In [38]:
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]
print(len(data9))
print(len(data12))

24611
82317


## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [39]:
k = 0.4
s = 0.9

In [40]:
# Si es necesario calcular el número de niños
data9['es_nino'] = data9['edad'] < 10
data12['es_nino'] = data12['edad'] < 10

data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data9['es_adulto'] = data9['edad'] > 10
data12['es_adulto'] = data12['edad'] > 10

data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [41]:
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [42]:
data9['ingr_t_t1'] = data9['ingr_t1_h'] / data9['escala']
data9['ingr_t_t2'] = data9['ingr_t2_h'] / data9['escala']
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']

data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [43]:
data9[['ingr_t_t1', 'ingr_t_t2', 'ingr_t_t3']].mean()

ingr_t_t1    96.331981
ingr_t_t2    97.456491
ingr_t_t3    99.700744
dtype: object

In [44]:
data12[['ingr_t_t4']].mean()

ingr_t_t4    81.469596
dtype: object

In [45]:
print("Ingreso individual descontando cargas familiares t4: ", data12['ingr_t_t4'].mean())
print("Mediana del ingreso individual descontando cargas familiares t4: ", data12['ingr_t_t4'].median())

Ingreso individual descontando cargas familiares t4:  81.46959571285619
Mediana del ingreso individual descontando cargas familiares t4:  62.09141744091688


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [46]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))
salario_dict = dict(zip(datos_actual['trimestre'], datos_actual['salario básico unificado']))
ano = 2003

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [60]:
resultados_list = []

# Para cada trimeste
for t in [1, 2, 3]:
    salario = salario_dict.get(t)
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    
    # Agrupa por región
    grouped = data9.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional = pd.DataFrame(resultados_list)
df_final_regional

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2003,1,Amazonía,0.305446,0.147294,0.103894,0.068332,0.149903,0.282986,105.339280,92.850881,129.91,1.399125
1,2003,1,Azuay,0.397874,0.180350,0.111709,0.072218,0.153459,0.267828,103.650005,83.318262,129.91,1.559202
2,2003,1,"Costa, Santo Domingo",0.519869,0.315611,0.232309,0.107500,0.224694,0.371765,84.007627,61.407192,129.91,2.115550
3,2003,1,El Oro,0.475003,0.212362,0.149321,0.077138,0.167619,0.312169,78.842623,68.500994,129.91,1.896469
4,2003,1,Guayas,0.591930,0.272936,0.173538,0.091116,0.184349,0.302044,74.574328,56.702057,129.91,2.291099
5,2003,1,Los Ríos,0.660289,0.357197,0.240839,0.079376,0.171991,0.311287,53.429201,46.622055,129.91,2.786449
6,2003,1,Manabí,0.611250,0.334893,0.231968,0.106092,0.221615,0.378791,70.323375,45.811156,129.91,2.835772
7,2003,1,Pichincha,0.189470,0.087765,0.058200,0.075459,0.153590,0.250393,164.788957,120.622978,129.91,1.076992
8,2003,1,Sierra,0.479365,0.246469,0.169199,0.083158,0.181805,0.337291,80.188672,66.948634,129.91,1.940443
9,2003,2,Amazonía,0.345318,0.157969,0.111639,0.072626,0.160164,0.304679,105.166461,92.226915,129.91,1.408591


In [61]:
resultados_list = []

# Para cada trimeste
for t in [4]:
    salario = salario_dict.get(t)
    col_ingr = f'ingr_t_t{t}'
    umbral = umbral_dict.get(t)
    
    # Agrupa por región
    grouped = data12.groupby('region')
    
    for region_name, group in grouped:
        # 1. Filtra datos
        valid = group.dropna(subset=[col_ingr])
        
        if len(valid) == 0:
            continue
            
        # Extrae los vectores 
        ingresos = valid[col_ingr].values
        pesos = valid['fexp'].values
        
        # 2. Calcula indices
        gaps = (umbral - ingresos) / umbral
        gaps = np.clip(gaps, a_min=0, a_max=None)
        
        # 3. Calcula FGT
        total_poblacion = pesos.sum()
        
        # FGT0
        fgt0 = (pesos * (gaps > 0).astype(int)).sum() / total_poblacion
        
        # FGT1
        fgt1 = (pesos * (gaps ** 1)).sum() / total_poblacion
        
        # FGT2
        fgt2 = (pesos * (gaps ** 2)).sum() / total_poblacion
        
        # 4. Calcula Ingreso promedio
        ingreso_promedio = np.average(ingresos, weights=pesos)

        # 5. Desigualdad de Atkinson
        atkinson_resultados = {}
        
        if ingreso_promedio > 0:
            for epsilon in [0.25, 0.5, 0.75]:
                # La suma ponderada de la utilidad
                utility_sum = np.sum((ingresos ** (1 - epsilon)) * pesos)
                
                # promedio de esa utilidad
                utility_mean = utility_sum / total_poblacion
                
                # ingreso equivalente
                y_ede = utility_mean ** (1 / (1 - epsilon))
                
                # índice final
                atkinson_index = 1 - (y_ede / ingreso_promedio)
                atkinson_resultados[f'a{int(epsilon*100)}'] = atkinson_index
        else:
            # Si nadie gana nada, definimos desigualdad como NaN
            atkinson_resultados = {'a25': np.nan, 'a50': np.nan, 'a75': np.nan}

        # 6. Calcula mediana del ingreso
        # Ordena
        sort_idx = np.argsort(ingresos)
        ingreso_ordenado = ingresos[sort_idx]
        pesos_ordenado = pesos[sort_idx]
        cumsum_pesos = np.cumsum(pesos_ordenado)
        cutoff = total_poblacion / 2.0
        mediana = ingreso_ordenado[np.searchsorted(cumsum_pesos, cutoff)]
        
        # 7. Guarda resultados
        resultados_list.append({
            'ano': ano,
            'trimestre': t,
            'region': region_name,
            'fgt0': fgt0,
            'fgt1': fgt1,
            'fgt2': fgt2,
            'a25': atkinson_resultados['a25'],
            'a50': atkinson_resultados['a50'],
            'a75': atkinson_resultados['a75'],
            'ingreso_promedio': ingreso_promedio,
            'ingreso_mediana': mediana + 1 if mediana < 1 else mediana,
            'salario_minimo': salario,
            'kaitz_indice': salario / (mediana + 1 if mediana < 1 else mediana)         
        })

# lista a DataFrame
df_final_regional1 = pd.DataFrame(resultados_list)
df_final_regional1

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2003,4,Amazonía,0.551462,0.279774,0.191606,0.088123,0.178690,0.280243,77.928881,62.615828,129.91,2.074715
1,2003,4,Azuay,0.398971,0.198901,0.125253,0.075910,0.150617,0.225020,97.595561,84.067009,129.91,1.545315
2,2003,4,"Costa, Santo Domingo",0.665867,0.327203,0.210735,0.093090,0.184558,0.284846,67.810753,49.861789,129.91,2.605402
3,2003,4,El Oro,0.382834,0.149764,0.081162,0.066337,0.126912,0.183554,97.388537,78.591386,129.91,1.652980
4,2003,4,Guayas,0.510462,0.236891,0.142927,0.106676,0.200376,0.286422,93.581855,65.314141,129.91,1.989003
5,2003,4,Los Ríos,0.563603,0.234527,0.132962,0.060268,0.117808,0.175623,71.780450,61.834245,129.91,2.100939
6,2003,4,Manabí,0.697879,0.378801,0.253760,0.088698,0.175742,0.263810,58.058953,43.050353,129.91,3.017629
7,2003,4,Pichincha,0.198241,0.065791,0.032195,0.073871,0.141921,0.204904,164.455845,119.570085,129.91,1.086476
8,2003,4,Sierra,0.602819,0.310059,0.206511,0.084438,0.170347,0.265197,68.496546,53.536276,129.91,2.426579


In [64]:
df_final = pd.concat([df_final_regional, df_final_regional1])
df_final

,ano,trimestre,region,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio,ingreso_mediana,salario_minimo,kaitz_indice
0,2003,1,Amazonía,0.305446,0.147294,0.103894,0.068332,0.149903,0.282986,105.339280,92.850881,129.91,1.399125
1,2003,1,Azuay,0.397874,0.180350,0.111709,0.072218,0.153459,0.267828,103.650005,83.318262,129.91,1.559202
2,2003,1,"Costa, Santo Domingo",0.519869,0.315611,0.232309,0.107500,0.224694,0.371765,84.007627,61.407192,129.91,2.115550
3,2003,1,El Oro,0.475003,0.212362,0.149321,0.077138,0.167619,0.312169,78.842623,68.500994,129.91,1.896469
4,2003,1,Guayas,0.591930,0.272936,0.173538,0.091116,0.184349,0.302044,74.574328,56.702057,129.91,2.291099
5,2003,1,Los Ríos,0.660289,0.357197,0.240839,0.079376,0.171991,0.311287,53.429201,46.622055,129.91,2.786449
6,2003,1,Manabí,0.611250,0.334893,0.231968,0.106092,0.221615,0.378791,70.323375,45.811156,129.91,2.835772
7,2003,1,Pichincha,0.189470,0.087765,0.058200,0.075459,0.153590,0.250393,164.788957,120.622978,129.91,1.076992
8,2003,1,Sierra,0.479365,0.246469,0.169199,0.083158,0.181805,0.337291,80.188672,66.948634,129.91,1.940443
9,2003,2,Amazonía,0.345318,0.157969,0.111639,0.072626,0.160164,0.304679,105.166461,92.226915,129.91,1.408591


### Inserta los cálculos en la base final

In [65]:
indices = pd.read_csv("indices_region.csv", encoding='latin-1')

In [66]:
import os

# 2. cheque el archivo
if not os.path.isfile('indices_region.csv'):
    # Headers si es la primera vez
    df_final.to_csv('indices_region.csv', index=False, encoding='latin-1')
else:
    # SI ya existe, append
    df_final.to_csv('indices_region.csv', mode='a', index=False, header=False, encoding='latin-1')